In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/Concrete_defects_CNN_v2"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")

APP_DIR = "/content/streamlit_app"
os.makedirs(os.path.join(APP_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(APP_DIR, "data"), exist_ok=True)

shutil.copy(
    os.path.join(CHECKPOINT_DIR, "resnet18_finetuned_best.pt"),
    os.path.join(APP_DIR, "checkpoints", "resnet18_finetuned_best.pt"),
)
shutil.copy(
    os.path.join(DATA_DIR, "best_thresholds.json"),
    os.path.join(APP_DIR, "data", "best_thresholds.json"),
)

print(os.listdir(os.path.join(APP_DIR, "checkpoints")))
print(os.listdir(os.path.join(APP_DIR, "data")))

Mounted at /content/drive
['resnet18_finetuned_best.pt']
['best_thresholds.json']


In [2]:
app_code = r'''
import json
import os

import numpy as np
import streamlit as st
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
import matplotlib.cm as cm

CHECKPOINT_PATH = "checkpoints/resnet18_finetuned_best.pt"
THRESHOLDS_PATH = "data/best_thresholds.json"
LABEL_COLS = ["NoDamage", "Crack", "Efflorescence", "Rust", "Spallation", "BarsExposed"]
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_resnet18(num_classes):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


@st.cache_resource
def load_model():
    model = build_resnet18(len(LABEL_COLS))
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    model = model.to(device)
    model.eval()
    return model


@st.cache_resource
def load_thresholds():
    with open(THRESHOLDS_PATH) as f:
        return json.load(f)


eval_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, class_idx]
        score.backward()

        pooled_gradients = self.gradients.mean(dim=[0, 2, 3])
        activations = self.activations[0]
        for i in range(activations.shape[0]):
            activations[i, :, :] *= pooled_gradients[i]

        heatmap = activations.mean(dim=0).cpu().numpy()
        heatmap = np.maximum(heatmap, 0)
        heatmap = heatmap / (heatmap.max() + 1e-8)
        return heatmap


def predict_and_explain(model, gradcam, img, thresholds):
    input_tensor = eval_transform(img).unsqueeze(0).to(device)
    input_tensor.requires_grad_()

    with torch.no_grad():
        probs = torch.sigmoid(model(input_tensor)).cpu().numpy()[0]

    predicted_classes = [
        LABEL_COLS[i] for i in range(len(LABEL_COLS)) if probs[i] > thresholds[LABEL_COLS[i]]
    ]
    if not predicted_classes:
        predicted_classes = ["NoDamage"]

    target_idx = int(np.argmax(probs))
    heatmap = gradcam.generate(input_tensor, target_idx)

    heatmap_resized = np.array(
        Image.fromarray((heatmap * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))
    )
    heatmap_colored = cm.jet(heatmap_resized / 255.0)[:, :, :3]

    img_resized = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    overlay = 0.5 * img_resized + 0.5 * heatmap_colored

    return {
        "probs": dict(zip(LABEL_COLS, probs.tolist())),
        "predicted": predicted_classes,
        "gradcam_class": LABEL_COLS[target_idx],
        "overlay": overlay,
        "heatmap": heatmap_resized,
    }


st.set_page_config(page_title="Concrete Defect Classifier", layout="wide")

st.title("Concrete Defect Classification")
st.caption(
    "AI-assisted preliminary inspection of concrete surfaces. "
    "Identifies visible defects to help prioritize detailed engineering inspection \u2014 "
    "does NOT determine structural safety or replace an engineer."
)

if not os.path.exists(CHECKPOINT_PATH):
    st.error(f"Model checkpoint not found at `{CHECKPOINT_PATH}`.")
    st.stop()

if not os.path.exists(THRESHOLDS_PATH):
    st.error(f"Thresholds file not found at `{THRESHOLDS_PATH}`.")
    st.stop()

model = load_model()
thresholds = load_thresholds()
gradcam = GradCAM(model, model.layer4[-1])

uploaded_file = st.file_uploader("Upload a concrete surface image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    img = Image.open(uploaded_file).convert("RGB")

    with st.spinner("Running inference..."):
        result = predict_and_explain(model, gradcam, img, thresholds)

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Original")
        st.image(img, use_container_width=True)
    with col2:
        st.subheader(f"Grad-CAM ({result['gradcam_class']})")
        st.image(result["overlay"], use_container_width=True, clamp=True)

    st.subheader("Predicted defects")
    st.write(", ".join(result["predicted"]))

    st.subheader("Per-class probabilities")
    for cls, p in result["probs"].items():
        t = thresholds[cls]
        flagged = "\u2705" if p > t else ""
        st.write(f"**{cls}**: {p:.3f}  (threshold: {t:.2f}) {flagged}")
        st.progress(min(float(p), 1.0))
else:
    st.info("Upload an image to get started.")
'''

with open("/content/streamlit_app/app.py", "w") as f:
    f.write(app_code)

print("app.py written,", len(app_code), "chars")

app.py written, 5124 chars


In [3]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.0 MB/s eta 0:00:00


In [13]:
process.terminate()  # stop the current instance first

import subprocess
import time

process = subprocess.Popen(
    ["streamlit", "run", "/content/streamlit_app/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    cwd="/content/streamlit_app",
    stdout=open("/content/streamlit_logs.txt", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(10)
print("Process running:", process.poll() is None)

Process running: True


In [14]:
!cat /content/streamlit_logs.txt



2026-09-08 20:49:44.717 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.197.10.165:8501

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [15]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [7]:
process.terminate()
print("Stopped.")

Stopped.
